# Step 5: LSTM — Long Short-Term Memory

The vanilla RNN's gradient either vanishes or explodes when backpropagating through
long sequences. The core problem: **every time step multiplies by the same matrix Wₕₕ**,
and repeated matrix multiplication either shrinks or amplifies the gradient exponentially.

The **LSTM** (Hochreiter & Schmidhuber, 1997) solves this with a **cell state** —
a separate memory lane that information can flow along with **additive updates** rather
than multiplicative recurrence:

```
cₜ = f ⊙ cₜ₋₁  +  i ⊙ g
```

The key: **addition**, not multiplication of the cell state across time steps.
This means the gradient of cₜ with respect to cₜ₋₁ is just `f` — a value
the network can keep near 1.0 whenever it needs to preserve information.

**What you'll see:**
1. LSTM equations: four gates, cell state, hidden state
2. Same character-level LM as the RNN — direct comparison
3. Gradient norms through time: LSTM vs. vanilla RNN
4. Long-range memory: LSTM tracking structure the RNN missed

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## The LSTM Equations

Four gates, each a sigmoid-gated linear combination of input and previous hidden state:

```
f = σ(Wf · [hₜ₋₁, xₜ] + bf)   # forget gate: what to erase from cell state
i = σ(Wi · [hₜ₋₁, xₜ] + bi)   # input gate:  how much of new candidate to add
g = tanh(Wg · [hₜ₋₁, xₜ] + bg) # candidate:   new information to potentially store
o = σ(Wo · [hₜ₋₁, xₜ] + bo)   # output gate: how much of cell state to expose

cₜ = f ⊙ cₜ₋₁ + i ⊙ g        # cell state: the long-range conveyor belt
hₜ = o ⊙ tanh(cₜ)              # hidden state: what's actually output
```

**The critical insight**: `cₜ = f ⊙ cₜ₋₁ + i ⊙ g` is an **additive** update.
The gradient of cₜ with respect to cₜ₋₁ is just `f`, which the network can set
close to 1.0 to let gradients flow unchanged across arbitrarily many steps.

In [ ]:
# Manual LSTM cell — one time step — to see the equations clearly
def lstm_cell_manual(x, h_prev, c_prev, Wf, Wi, Wg, Wo, bf, bi, bg, bo):
    """
    Single LSTM step. All W matrices have shape (hidden + input, hidden).
    x: (input_size,), h_prev: (hidden,), c_prev: (hidden,)
    """
    combined = np.concatenate([h_prev, x])  # concatenate hidden + input

    def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    f = sigmoid(Wf @ combined + bf)         # forget gate ∈ (0,1)
    i_gate = sigmoid(Wi @ combined + bi)    # input gate ∈ (0,1)
    g = np.tanh(Wg @ combined + bg)         # candidate ∈ (-1,1)
    o = sigmoid(Wo @ combined + bo)         # output gate ∈ (0,1)

    c_new = f * c_prev + i_gate * g         # additive cell state update
    h_new = o * np.tanh(c_new)             # bounded hidden state

    return h_new, c_new, {'f': f, 'i': i_gate, 'g': g, 'o': o}

# Demonstrate: a cell that remembers a value for a long time
hidden_size, input_size = 1, 1
# Hand-set weights: forget gate ≈ 1 (remember), input gate ≈ 0 (no new input)
W = np.zeros
# Simple demo: feed in a spike at step 0, then zeros; observe cell state
h, c = np.zeros(hidden_size), np.zeros(hidden_size)
Wf = np.array([[5.0, 0.0]])   # high forget gate
Wi = np.array([[5.0, 5.0]])
Wg = np.array([[0.0, 5.0]])
Wo = np.array([[5.0, 0.0]])   # expose cell when h is nonzero
bf = np.array([0.0])
bi = np.array([-3.0])  # low input gate by default
bg = np.array([0.0])
bo = np.array([0.0])

cell_states = [0.0]
gate_vals = []
inputs = [1.0] + [0.0] * 19  # spike at step 0, then zeros

for x_val in inputs:
    x = np.array([x_val])
    h, c, gates = lstm_cell_manual(x, h, c, Wf, Wi, Wg, Wo, bf, bi, bg, bo)
    cell_states.append(c[0])
    gate_vals.append({k: v[0] for k, v in gates.items()})

print("Cell state over 20 steps after a spike at step 0:")
print([f"{v:.3f}" for v in cell_states[:10]], "...")

## Same Task as the RNN: Character-Level Language Model

In [ ]:
# Same corpus as notebook 04
text = """To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them to die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished to die to sleep
To sleep perchance to dream aye theres the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause there is the respect
That makes calamity of so long life"""

chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
data = torch.tensor([char2idx[c] for c in text], dtype=torch.long)

class CharLM(nn.Module):
    """Character-level language model. Swap rnn_type to compare RNN vs LSTM."""

    def __init__(self, vocab_size, hidden_size, rnn_type='lstm'):
        super().__init__()
        self.rnn_type = rnn_type
        self.embed = nn.Embedding(vocab_size, hidden_size)
        if rnn_type == 'lstm':
            self.rnn = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        else:
            self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True,
                              nonlinearity='tanh')
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        x = self.embed(x)          # (batch, seq, hidden)
        out, state = self.rnn(x, state)
        logits = self.fc(out)      # (batch, seq, vocab)
        return logits, state

    def generate(self, seed, length=200, temperature=1.0):
        self.eval()
        generated = list(seed)
        x = torch.tensor([[char2idx[c] for c in seed]], device=device)
        state = None
        with torch.no_grad():
            logits, state = self.forward(x, state)
            for _ in range(length):
                probs = torch.softmax(logits[0, -1] / temperature, 0)
                idx = torch.multinomial(probs, 1).item()
                generated.append(idx2char[idx])
                x = torch.tensor([[idx]], device=device)
                logits, state = self.forward(x, state)
        return ''.join(generated)


def train_model(model, data, n_epochs=500, seq_len=40, lr=3e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    grad_norms = []
    for epoch in range(n_epochs):
        model.train()
        start = np.random.randint(0, len(data) - seq_len - 1)
        x = data[start:start+seq_len].unsqueeze(0).to(device)
        y = data[start+1:start+seq_len+1].unsqueeze(0).to(device)

        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()

        # Measure gradient norm on recurrent weights
        if model.rnn_type == 'lstm':
            g = model.rnn.weight_hh_l0.grad
        else:
            g = model.rnn.weight_hh_l0.grad
        if g is not None:
            grad_norms.append(g.norm().item())

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    return losses, grad_norms

hidden_size = 128
lstm_model = CharLM(vocab_size, hidden_size, 'lstm').to(device)
rnn_model  = CharLM(vocab_size, hidden_size, 'rnn').to(device)

print("Training LSTM...")
lstm_losses, lstm_grad_norms = train_model(lstm_model, data, n_epochs=500)
print("Training vanilla RNN...")
rnn_losses,  rnn_grad_norms  = train_model(rnn_model,  data, n_epochs=500)

In [ ]:
def smooth(arr, w=30):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Training loss comparison
ax = axes[0]
ax.plot(smooth(lstm_losses), 'steelblue', lw=2, label='LSTM')
ax.plot(smooth(rnn_losses),  'tomato',    lw=2, label='Vanilla RNN')
ax.axhline(np.log(vocab_size), color='k', linestyle=':', lw=1, label='random chance')
ax.set_xlabel('Training step'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('LSTM vs. Vanilla RNN — same task, same data')
ax.legend(); ax.grid(True, alpha=0.3)

# Gradient norm comparison
ax = axes[1]
if lstm_grad_norms and rnn_grad_norms:
    ax.semilogy(smooth(lstm_grad_norms, 20), 'steelblue', lw=2, label='LSTM Wₕₕ')
    ax.semilogy(smooth(rnn_grad_norms,  20), 'tomato',    lw=2, label='RNN Wₕₕ')
    ax.set_xlabel('Training step'); ax.set_ylabel('Gradient norm (log)')
    ax.set_title('Recurrent weight gradient norms')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('05_lstm_vs_rnn.png', bbox_inches='tight')
plt.show()

In [ ]:
# Visualize gate activations — what the LSTM is doing internally
lstm_model.eval()
seed = "To be"
x = torch.tensor([[char2idx[c] for c in seed]], device=device)

# Run the LSTM and capture internal gate values
gate_data = []
with torch.no_grad():
    emb = lstm_model.embed(x)
    # Manually run the LSTM cell step by step to extract gates
    W_ih = lstm_model.rnn.weight_ih_l0  # (4*hidden, input)
    W_hh = lstm_model.rnn.weight_hh_l0  # (4*hidden, hidden)
    b_ih = lstm_model.rnn.bias_ih_l0    # (4*hidden,)
    b_hh = lstm_model.rnn.bias_hh_l0    # (4*hidden,)
    H = lstm_model.rnn.hidden_size

    h = torch.zeros(1, H, device=device)
    c = torch.zeros(1, H, device=device)
    for t in range(emb.size(1)):
        xt = emb[0, t]  # (H,)
        gates = xt @ W_ih.T + b_ih + h[0] @ W_hh.T + b_hh  # (4H,)
        fi, ii, gi, oi = gates.chunk(4)
        f_gate = torch.sigmoid(fi).mean().item()
        i_gate = torch.sigmoid(ii).mean().item()
        g_gate = torch.tanh(gi).abs().mean().item()
        o_gate = torch.sigmoid(oi).mean().item()
        gate_data.append({'f': f_gate, 'i': i_gate, 'g': g_gate, 'o': o_gate})
        c = torch.sigmoid(fi) * c + torch.sigmoid(ii) * torch.tanh(gi)
        h = torch.sigmoid(oi) * torch.tanh(c)

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = range(len(seed))
width = 0.2
colors = {'f': 'tomato', 'i': 'steelblue', 'g': 'green', 'o': 'purple'}
labels = {'f': 'Forget', 'i': 'Input', 'g': '|Candidate|', 'o': 'Output'}
for k, (name, color) in enumerate(colors.items()):
    vals = [d[name] for d in gate_data]
    offset = (k - 1.5) * width
    ax.bar([p + offset for p in x_pos], vals, width, label=labels[name], color=color, alpha=0.8)

ax.set_xticks(list(x_pos)); ax.set_xticklabels(list(seed))
ax.set_xlabel('Input character'); ax.set_ylabel('Average gate activation')
ax.set_title(f'LSTM gate activations for input: {repr(seed)}')
ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('05_lstm_gates.png', bbox_inches='tight')
plt.show()

In [ ]:
print("=== Generated Text (LSTM) ===")
for temp in [0.7, 1.0]:
    print(f"\nTemperature {temp}:")
    print(lstm_model.generate('To', length=200, temperature=temp))

print("\n=== Generated Text (Vanilla RNN) ===")
for temp in [0.7, 1.0]:
    print(f"\nTemperature {temp}:")
    print(rnn_model.generate('To', length=200, temperature=temp))

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Cell state** | Additive update; gradient can flow unchanged across many steps |
| **Forget gate** | Learns when to clear the cell (f=0) vs. keep it (f=1) |
| **Input gate** | Controls how much new information enters the cell |
| **Output gate** | Controls what fraction of the cell state to expose as hidden state |
| **Gradient highway** | ∂cₜ/∂cₜ₋₁ = f — network can set f≈1 to prevent vanishing |
| **Better gradients** | LSTM Wₕₕ gradients more stable than vanilla RNN |

**What's still missing**: both RNN and LSTM process sequences **one step at a time**.
For a sequence of length T, encoding requires T serial steps — no parallelism.
And even LSTMs struggle with very long-range dependencies.

Next: **Seq2Seq** — using two LSTMs (encoder + decoder) to map variable-length
sequences to variable-length outputs.